# Task 1: Preprocessing

**Input:** `data/raw/reviews_raw.csv` — raw scraped reviews  
**Output:** `data/clean/reviews_clean.csv` — translated, deduplicated, normalized dataset

**Steps:**
1. Load raw data and inspect
2. Translate Amharic reviews to English (emojis preserved)
3. Remove duplicate reviews
4. Handle missing values
5. Normalize dates to `YYYY-MM-DD`
6. Final quality check and export

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from scripts.preprocess import translate_amharic, preprocess_reviews

## 1. Load Raw Data

In [ ]:
raw_df = pd.read_csv("../data/raw/reviews_raw.csv")

print(f"Shape: {raw_df.shape}")
print(f"\nColumns: {list(raw_df.columns)}")
print(f"\nDtypes:\n{raw_df.dtypes}")
raw_df.head()

## 2. Pre-cleaning Snapshot

In [ ]:
print("=== Reviews per bank (raw) ===")
print(raw_df["bank"].value_counts())

print("\n=== Null values per column ===")
print(raw_df.isnull().sum())

print("\n=== Duplicate review IDs ===")
print(f"Total duplicates: {raw_df.duplicated(subset='id').sum()}")

## 3. Translate Amharic Reviews

Many users write reviews in Amharic. We detect these using `langdetect` and translate them to English using `deep-translator`'s Google Translate backend. Emojis are preserved since they carry sentiment signal.

Only reviews detected as Amharic (`lang == 'am'`) are sent to the translation API — English reviews are left untouched.

In [ ]:
translated_df = translate_amharic(raw_df.copy())
translated_df.head()

## 4. Clean & Normalize

Calling `preprocess_reviews()` from `scripts/preprocess.py` which:
- Drops duplicate `id` entries
- Drops rows where `review` or `rating` is null
- Normalizes `date` to `YYYY-MM-DD`
- Returns only the five required columns: `review`, `rating`, `date`, `bank`, `source`

In [ ]:
clean_df = preprocess_reviews(translated_df)
clean_df.head()

## 5. Post-cleaning Quality Check

In [ ]:
print("=== Reviews per bank (clean) ===")
print(clean_df["bank"].value_counts())

print("\n=== Missing values ===")
print(clean_df.isnull().sum())

print("\n=== Date range ===")
print(f"Earliest: {clean_df['date'].min()}")
print(f"Latest:   {clean_df['date'].max()}")

print("\n=== Rating distribution ===")
print(clean_df["rating"].value_counts().sort_index())

print(f"\n=== Total reviews: {len(clean_df)} ===")

## 6. Export Clean Dataset

Saved to `data/clean/reviews_clean.csv`. Listed in `.gitignore` — will not be committed to GitHub.

In [ ]:
os.makedirs("../data/clean", exist_ok=True)

# Save individual bank files
for bank_name, group in clean_df.groupby("bank"):
    filename = bank_name.lower().replace(" ", "_") + "_clean.csv"
    group.to_csv(f"../data/clean/{filename}", index=False)
    print(f"Saved {len(group)} reviews → data/clean/{filename}")

# Save combined file for use in tasks 2, 3, and 4
clean_df.to_csv("../data/clean/reviews_clean.csv", index=False)
print(f"\nSaved combined {len(clean_df)} reviews → data/clean/reviews_clean.csv")